# Final Project

ST554: Analysis of Big Data Spring 2026

Ryan Mersereau

## Goals

For this project, we'll use spark to handle streaming data and fit a machine learning model, among other things.

The data is modified from the UCI machine learning repository. The study was about relating power
consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.
* We’ll have a chunk of the (modified) data to build a model on.
* We will then ‘stream data’ to a folder in the form of CSV files. We'll be monitoring this folder. When
data comes in and use a fitted model to predict on the incoming data!

## Fitting the Model

The file `power_ml_data.csv`comes from [here](https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv), we will read this data into a standard pandas dataframe and convert this into a spark data frame.

First, let's import needed modules and start our Spark Session

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Binarizer, OneHotEncoder, StringIndexer,
    VectorAssembler, PCA, SQLTransformer
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("PowerZoneFinal").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 13:18:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 13:18:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 13:18:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/29 13:18:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/29 13:18:31 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [2]:
# Read data into pandas, convert to spark df
url = "https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv"
pdf = pd.read_csv(url)

df = spark.createDataFrame(pdf)
df.printSchema()
df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We can see our dataframe has 10 columns. Here we'll be using `Power_Zone_3` as our response variable, and all other variables as predictors (in case `Power_Zone_3` were to go offline).

We want to fit an elastic net model using CV, and will start by performing some transformations using `MLlib` functions that can be put in a pipeline.

### Transformations

To start, we need to cast our`Hour`variable as`DoubleType`(currently`long`) and then binarize this column into night vs day.

In [3]:
sql_transformer = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__"
)

# binarize hour (threshold = 6.5, night = 0, day = 1)
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_double",
    outputCol="Hour_bin"
)

Next, we need to One-hot encode the`Month`column. This also converts these month categories into binary format by creating separate column for each category.

In [4]:
ohe = OneHotEncoder(
    inputCols=["Month"],
    outputCols=["Month_ohe"],
    dropLast=True   
)

Now lets run a PCA (Principal component analysis) fit on the `Temperature`,`Humidity`,`Wind_Speed`,`General_Diffuse_Flows`, and`Diffuse_Flows`columns. 

We'll do this by placing these variables in a column together with a `VectorAssembler()`and then use this with the `PCA()`estimator. We'll use two PCs in our transformation.

In [5]:
weather_cols = ["Temperature", "Humidity", "Wind_Speed",
                "General_Diffuse_Flows", "Diffuse_Flows"]

weather_assembler = VectorAssembler(
    inputCols=weather_cols,
    outputCol="weather_features"
)

# Fit PCA with 2 components
pca = PCA(k=2, inputCol="weather_features", outputCol="pca_features")

Now, we can rename the response variable (`Power_Zone_3`) as`label` and use`VectorAssembler()`to put our predictors as`features`.

We will use the:
* two fitted PCA features
* binary `Hour` variable
* `Power_Zone_1`
* `Power_Zone_2`
* `Month` indicator variables

In [6]:
label_transformer = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

In [7]:
# VectorAssembler
feature_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

### Pipeline

Now, we will combine all our prior transformations together to build a pipeline.

In [8]:
pipeline = Pipeline(stages=[
    sql_transformer,    
    binarizer,          
    ohe,                
    weather_assembler,  
    pca,                
    label_transformer,  
    feature_assembler   
])

### CrossValidator with ElasticNet

Now, we'll use the`CrossValidator()`and`LinearRegression()`functions to fit an elastic net model.

We'll build a parameter grid all combinations of the following:
∗ regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
∗ elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

Then, we can fit the model using 5-fold CV with`rmse`as our criterion.

In [9]:
import logging

#Suppress noisy MLlib warnings
logging.getLogger("py4j").setLevel(logging.ERROR)
spark.sparkContext.setLogLevel("ERROR")

lr = LinearRegression(featuresCol="features",
                      labelCol="label",
                      solver = "l-bfgs") # Avoids warning of failed Cholesky solver due do singular matrix

# Full parameter grid (11 x 11 = 121 combinations)
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,         [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam,  [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# Merged pipeline stages and lr model into one combined pipeline
cv = CrossValidator(
    estimator=Pipeline(stages=[*pipeline.getStages(), lr]),
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=7,
    parallelism = 8 # 128 gave me a memory error
)

In [10]:
# Suppress "plan too large" warning
spark.conf.set("spark.sql.debug.maxToStringFields", "200")

# Fit the model
cv_model = cv.fit(df)

After our model finishes running (took me ~30 minutes, maybe could speed up by slowly increasing `parallelism`?), we can report the optimal values chosen for the tuning parameters and the cv error.

In [11]:
best_lr = cv_model.bestModel.stages[-1]  # last stage is the LinearRegression

print(f"Optimal regParam:        {best_lr._java_obj.getRegParam()}")
print(f"Optimal elasticNetParam: {best_lr._java_obj.getElasticNetParam()}")
print(f"CV RMSE (best):          {min(cv_model.avgMetrics):.4f}")

Optimal regParam:        0.25
Optimal elasticNetParam: 0.25
CV RMSE (best):          2147.4545


We can see the optimal values chosen for `regParam`and`elasticNetParam` are both 0.25, and the CV error is 2147.45

Now, lets report the training set RMSE using the fitted model as a transformer and evaluating on the entire training set. 

In [13]:
predictions = cv_model.transform(df)

train_rmse = evaluator.evaluate(predictions)
print(f"Training Set RMSE: {train_rmse:.4f}")

Training Set RMSE: 2147.0977


Finally, we can take the the outputted transformations from the model (predictions) and create a`residual`column (`label`-`prediction`) using `.withColumn()` method. We'll print out a dataframe with these `residuals`, `label` column, and `predictions`

In [14]:
from pyspark.sql.functions import col

residuals_df = predictions.withColumn(
    "residual", col("label") - col("prediction")
)

residuals_df.select("label", "prediction", "residual").show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20881.05459129693|-640.0907312969284|
|20131.08434|18659.261051013502|1471.8232889864994|
|19668.43373| 18203.76804964944|1464.6656803505612|
|18899.27711|17589.731323831085|1309.5457861689138|
|18442.40964|  16996.3667771145|1446.0428628855007|
|18130.12048|16516.774276381617|1613.3462036183846|
|17945.06024|16092.365738290788|1852.6945017092112|
|17459.27711|15721.820068858538|1737.4570411414606|
|17025.54217|15270.192957709256|1755.3492122907446|
|16794.21687|14937.496415540296|1856.7204544597043|
|16638.07229|14651.686481907414|1986.3858080925856|
|16395.18072|14414.212770012422|1980.9679499875783|
|16117.59036|14082.050732717555|2035.5396272824455|
| 15822.6506|13624.080137615347| 2198.570462384654|
|15672.28916|13449.621866572135| 2222.667293427865|
|15597.10843|13301.642354139087|2295.4660758609134|
|15510.36145

## Streaming

Now that the model is fitted, we'll move on to streaming data with spark, using the file found [here.](https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv)

We'll be randomly sampling rows from this to output to `.csv` files that will be read in, so its also downloaded locally in the Jupyter environment so our `.py` file can find it!

**Before** actually starting the stream query, we'll need to ensure our python file is loaded in a separate console so that csv files land where we want them to.

### Reading a Stream

A folder `power_stream_folder` was created to send our`.csv`files to. Now, let's set up the schema for the stream using schema from the original data

In [15]:
stream_schema = df.schema
print("Stream schema:")
stream_schema

Stream schema:


StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

Next, we'll set up the `readStream`. We point Spark at the watch folder with `header=True` so it skips the column names that our python file writes in.

In [16]:
power_stream = (
    spark.readStream
         .schema(stream_schema)
         .option("header", True)
         .csv("power_stream_folder")
)

### Transformation and Aggregation

Now, we'll perform two separate transformations on the stream object and join them on the `label` (response) column:

* Obtain predictions from the incoming data: Do this by applying `cv_model` and computing `residual = label - prediction`, keeping only those three columns
* Modify the response variable (`Power_Zone_3`) to be renamed to`label`, so we have a key to join on.

In [17]:
# 1st transformation: apply the fitted cross-validated model to the stream
pred_stream = cv_model.transform(power_stream)

# Compute residual and keep only the three required columns
pred_stream = (
    pred_stream
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)

# 2nd transformation: renames response to 'label'
label_stream = power_stream.withColumn("label", col("Power_Zone_3"))

# Both DataFrames originate from the same underlying stream, so we can
# join them directly.  We use an inner join on 'label' as instructed.
joined_stream = pred_stream.join(label_stream, on="label", how="inner")

### Writing Step

Now, its time to write the stream to the console and start the query. We'll use`append`output mode

In [18]:
stream_query = (
    joined_stream
    .writeStream
    .outputMode("append")
    .format("console")
    .start()
)

print("Stream query started.  Run stream_producer.py in a separate console.")
print(f"Query name : {stream_query.name}")
print(f"Query id   : {stream_query.id}")

Stream query started.  Run stream_producer.py in a separate console.
Query name : None
Query id   : 19c20c5a-9b40-4d6b-a298-860b684d6b68


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|7571.668667| 7227.805904660596| 343.86276233940407|       8.67|    86.9|     0.084|                47.95|        34.46| 24292.01521| 18377.41639| 7571.668667|   12|   8|
|17408.25911| 17702.42957775678|-294.17046775677954|      20.97|    75.3|     4.927|                284.7|        288.9| 34043.80328| 20942.41486| 17408.25911|    5|  17|
|14531.30699|12495.764463940872| 2035.5425260591

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|13282.43161|11770.639364010975|  1511.792245989025|      20.55|   65.66|     4.923|                0.102|        0.056|  28573.1291| 16248.54772| 13282.43161|   10|   0|
|14794.83871|13536.423561171385| 1258.4151488286152|       9.64|    87.4|     0.087|                0.037|        0.115| 23248.34043| 13613.41463| 14794.83871|    3|   5|
|12226.02656|12289.099677457009|-63.073117457008

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|24918.64615|27424.141546644587|-2505.4953966445864|      21.98|    75.6|      0.07|                172.4|        174.6| 45863.84106| 26816.63202| 24918.64615|    6|  19|
|18061.21457| 19918.25804059381|-1857.0434705938096|      18.77|    79.5|     0.069|                 41.3|        33.48| 35416.13115|  22075.5418| 18061.21457|    5|  18|
|28156.06154|29514.248748549147|-1358.1872085491

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|16612.25806| 18090.10328785281|-1477.8452278528093|      15.53|   59.88|     0.076|                737.0|         93.3|     35424.0|  21190.2439| 16612.25806|    3|  13|
|15522.90909| 15138.07468405728| 384.83440594271997|       15.6|    82.9|     0.073|                0.022|        0.163| 23598.01938| 11144.60285| 15522.90909|    4|   2|
|28197.48954|29013.685562761177|  -816.196022761

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|26771.66144|25740.027910175228| 1031.6335298247723|       26.6|   66.36|     0.068|                792.0|         62.9| 40422.28635| 28199.36642| 26771.66144|    8|  14|
| 25396.1005|25364.659361703427| 31.441138296573627|      12.81|    74.3|     4.917|                1.976|        2.018| 43016.94915|  25710.6383|  25396.1005|    2|  19|
|15307.63636|15373.286974203034| -65.6506142030

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|16538.91213|21630.529500686407|-5091.6173706864065|      22.65|   67.77|     4.923|                200.2|        180.1| 26988.43854|     18600.0| 16538.91213|    7|   7|
|15696.51861|17932.423229759217| -2235.904619759218|      15.82|    73.6|     0.073|                3.476|        3.448| 39069.20152| 32995.39736| 15696.51861|   12|  18|
|19369.24012|22136.558164920807|-2767.318044920

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|14472.36181| 14218.97659288091| 253.38521711909016|      13.43|    71.1|     0.075|                0.029|        0.119|  23637.9661| 14779.33131| 14472.36181|    2|   6|
|26081.92771|25659.771572598675|  422.1561374013254|      14.64|   57.74|     4.921|                0.088|        0.108|  43175.6962| 25572.03647| 26081.92771|    1|  20|
|17576.72727|18595.659846827326|-1018.932576827

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|18685.02024|   18231.453236963|   453.567003037002|      22.62|   56.12|     0.073|                911.0|        50.58| 35844.19672| 22105.26316| 18685.02024|    5|  13|
|11490.82067| 9420.059033616562|  2070.761636383437|      18.59|    81.7|     4.915|                0.033|        0.148| 24804.55142| 15613.69295| 11490.82067|   10|   3|
|21460.08097|22146.005147562446| -685.924177562

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|14866.39676| 13076.36227287316|  1790.034487126839|      22.55|    67.8|     0.073|                476.9|        301.0| 28038.29508| 15971.51703| 14866.39676|    5|  11|
|15769.08543| 14879.95141864051|  889.1340113594888|      16.16|   58.19|     0.079|                290.6|        299.3| 28696.27119| 22672.34043| 15769.08543|    2|  16|
|28625.45455|27312.028421357227| 1313.426128642

Finally, after all 20 batches have been sent, we can **stop** the stream.

In [19]:
stream_query.stop()
print("Stream stopped.")

Stream stopped.
